# GazeFormer-MoE: feature extractor evaluation

**Not run.** The repo releases no pretrained weights and no inference code, so there is no env and nothing below section 2 has been executed. The model card and notes come from reading the repo at the commit below.

## Notes and verdict

**Hardware used for the speed numbers:** n/a (not run)

**Status:** inspected only; no env, no weights, not run on `test_pictures`.

**Strengths:**
- Gaze-only model, which is what we'd want it for. Reports cross-dataset training/testing on Gaze360, ETH-XGaze, MPIIFaceGaze, and EyeDiap (README).
- README badge says MIT, which would be a permissive license if confirmed.

**Weaknesses:**
- **No pretrained weights.** No checkpoint link in the README or code, no GitHub releases. `config.py` names `epoch_30.pth`, but nothing loads it. Using it means training it ourselves on a gaze dataset.
- **No inference code.** No demo or single-image script; the only evaluation is the validation loop inside `train.py`, over GazeHub-format datasets.
- **Incomplete code.** `moba_attn_zhao` is imported but not in the repo (optional, wrapped in try/except). Issue #1, "缺少部分代码模块" ("some code modules are missing"), reports the same.
- **No requirements file.** The README only lists what one "should contain"; `flash_attn`, tensorboard, and matplotlib are imported but not listed. The install instructions point to a placeholder repo (`your-username/Gazeformer_submission`).
- **flash-attn by default.** Attention layers default to `use_flash_attn=True`; flash-attn 2 needs an Ampere or newer GPU, and our RTX 2080 SUPER is Turing (compute capability 7.5). Turning it off means changing how the model is built.
- **Input needs normalization we don't have.** Trained on GazeHub-normalized face crops (224×224, `[0.5, 0.5, 0.5]` mean/std), which needs landmarks and head-pose-based normalization that this repo doesn't provide.
- **License unclear:** MIT badge in the README, but no LICENSE file and GitHub detects no license.
- Paper not linked from the repo; `clipgaze.pdf` is only the architecture figure.
- Gaze only: no AUs, no head pose.

**Verdict:** drop (cannot evaluate)

**Why:** Without released weights or inference code it can't be run on our clips like the other models. Revisit only if the authors release a checkpoint.

## Setup

Imports and paths. Run this first.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()

from src import clip_io, plots, schema

MODEL = 'GazeFormer-MoE'
MODEL_DIR = ROOT / 'models' / MODEL

## 1. Model card

| Field | Value |
|---|---|
| Paper | https://arxiv.org/pdf/2605.00799 |
| Repo URL | https://github.com/AIPMLab/GazeFormer-MoE |
| Commit hash | `10d96efc6cd8ccc6a0c412f445dde3cd43b7cb05` (2026-05-27) |
| License | Unclear: README badge says MIT; no LICENSE file; GitHub detects none |
| Weights | **None released** (no link in README or code, no GitHub releases) |
| Pipeline | Face image → three streams: frozen CLIP ViT-B/32 (image embedding fused with the best-matching text prompts for illumination, background, head pose, gaze direction), CNN backbone (ResNet-50 by default), CLIP patch tokens → Mixture-of-Experts transformer → 3D gaze vector |
| Inputs required | Normalized face crop as in GazeHub (224×224, RGB, mean/std 0.5); face detection and normalization not included |
| Outputs | 3D gaze vector (`nn.Linear(256, 3)` head in `models.py`). No AUs, no head pose |
| Native units / conventions | Unit gaze vector in the normalized camera frame; `train.py` converts it with pitch = asin(y), yaw = atan2(x, z), in radians. Sign directions relative to the subject not checked |
| Conversion to schema | Not done (not run). Would be: vector → pitch/yaw → degrees, signs to be verified |
| Env | None created |

In [ ]:
!git -C {MODEL_DIR} remote get-url origin
!git -C {MODEL_DIR} rev-parse HEAD

## 2. Setup log

Not set up: the repo has no pretrained weights or inference code (see notes).

| # | Step / command | Result | Error | Fix |
|---|---|---|---|---|
| 1 | Inspect repo (README, `config.py`, imports, GitHub releases and issues) | No weights, no inference script, no requirements file; `flash_attn` needs Ampere+ | | Stopped before creating an env |

## 3. Run on test clips

**Not run** (no weights). Cells below are the unchanged template.

`predict(image)` is the only model-specific code. It takes one BGR frame and returns `(face_conf, outputs)`: `outputs` is a dict of schema column names to values, already converted to schema units and signs, or `None` when no face is found. List every column it can return in `OUTPUT_COLUMNS`.

In [ ]:
# TODO: load the model
model = None

In [ ]:
# TODO: every column predict() can return, in a fixed order.
# Frames with no face get NaN in all of them, so clips with no detections still save.
OUTPUT_COLUMNS = []

def predict(image):
    """TODO: run the model on one BGR frame.

    Returns (face_conf, outputs). outputs is e.g.
    {'AU12_int': 1.3, 'gaze_yaw_deg': -4.2, ...} or None if no face.
    """
    raise NotImplementedError

In [ ]:
TEST_SET = 'test_pictures'  # folder under data/ with manifest.csv
manifest = clip_io.load_manifest(TEST_SET)
manifest

In [ ]:
results, timings = {}, {}
for row in manifest.itertuples():
    records = []
    t0 = time.perf_counter()
    for frame, timestamp_ms, image in clip_io.iter_frames(clip_io.clip_path(row)):
        face_conf, outputs = predict(image)
        record = {'clip_id': row.clip_id, 'frame': frame, 'timestamp_ms': timestamp_ms,
                  'face_conf': float(face_conf) if outputs is not None else 0.0}
        record.update(outputs or {})
        records.append(record)
    timings[row.clip_id] = time.perf_counter() - t0
    results[row.clip_id] = pd.DataFrame.from_records(records, columns=list(schema.REQUIRED_COLUMNS) + OUTPUT_COLUMNS)
    print(f'{row.clip_id}: {len(records)} frames in {timings[row.clip_id]:.1f} s')

## 4. Save output in the common schema

`save_output` runs `schema.validate` and refuses to write a file that fails it. Fix the conversion in `predict`, not the check.

In [ ]:
for clip_id, df in results.items():
    df = df.astype({'frame': 'int64', 'timestamp_ms': 'float64', 'face_conf': 'float64'})
    out_cols = [c for c in df.columns if c not in schema.REQUIRED_COLUMNS and not c.startswith(schema.EXTRA_PREFIX)]
    df[out_cols] = df[out_cols].astype('float64')
    df = schema.add_missing_au_pairs(df)
    print(clip_io.save_output(df, MODEL, clip_id))

## 5. Quick checks

These read back the saved files, so they test exactly what was written.

In [ ]:
out = clip_io.load_outputs(MODEL)
print('output columns:', schema.output_columns(out))

### Face-detection rate

In [ ]:
display(plots.detection_rate(out).to_frame())
plots.plot_detection_rate(out);

### Speed (fps on this machine)

Includes video decoding. Note the hardware (CPU/GPU) in the verdict.

In [ ]:
plots.fps_table(timings, out)

### Standard plots per clip

In [ ]:
for clip_id, df in out.groupby('clip_id'):
    plots.plot_face_conf(df)
    if schema.au_ids(df):
        for kind in ('int', 'occ'):
            if df.filter(like=f'_{kind}').notna().any().any():
                plots.plot_aus(df, kind)
    if any(c in df for c in schema.GAZE_COLUMNS):
        plots.plot_gaze(df)
    if any(c in df for c in schema.POSE_COLUMNS):
        plots.plot_pose(df)
    plt.show()

### Plausibility and stability (no ground truth)

- Value ranges: do angles and AU levels look reasonable for what happens in each clip?
- Jitter: median absolute frame-to-frame change on detected frames. A high value on a still face means a noisy model.
- AU occurrence rate: AUs that are always on or always off are suspicious.

In [ ]:
detected = out[out['face_conf'] >= plots.DEFAULT_CONF_THRESHOLD]
display(detected[schema.output_columns(out)].describe().T)

angle_cols = [c for c in schema.GAZE_COLUMNS + schema.POSE_COLUMNS if c in out]
if angle_cols:
    jitter = detected.groupby('clip_id')[angle_cols].apply(lambda d: d.diff().abs().median())
    display(jitter.rename(columns=lambda c: c.replace('_deg', '_jitter_deg')))

occ_cols = [c for c in out if c.endswith('_occ') and out[c].notna().any()]
if occ_cols:
    display(detected.groupby('clip_id')[occ_cols].mean())

### Failure cases

Frames where the face was missed or detected with low confidence. Look at a few to see why (pose, occlusion, lighting, blur).

In [ ]:
failures = plots.failure_frames(out)
display(failures.groupby('clip_id').size().rename('n_failures').to_frame())
failures.head(20)

In [ ]:
def show_frames(clip_id, frames, ncols=4):
    row = manifest.set_index('clip_id').loc[clip_id]
    wanted = set(frames)
    images = [(f, img) for f, _, img in clip_io.iter_frames(clip_io.clip_path(row), rgb=True) if f in wanted]
    if not images:
        return
    nrows = -(-len(images) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 2.5 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (f, img) in zip(axes.flat, images):
        ax.imshow(img)
        ax.set_title(f'{clip_id} frame {f}', fontsize='small')
    plt.show()

# Up to 8 evenly spaced failure frames per clip
for clip_id, f in failures.groupby('clip_id'):
    show_frames(clip_id, f['frame'].iloc[np.linspace(0, len(f) - 1, min(8, len(f))).astype(int)])